# Kerr Black Hole

A rapidly rotating Kerr black hole twists the surrounding spacetime itself, producing the phenomenon known as frame dragging. In this visualization, the glowing warped rings represent the distorted structure of spacetime and magnetic fields near the event horizon, while the bright accretion disk marks superheated matter spiraling inward at relativistic speeds. The central dark region is the black hole’s shadow, surrounded by the photon ring — light bent into orbit by extreme gravity. Polar jets emerge along the rotation axis, carrying enormous amounts of energy far into interstellar space.

In [2]:
from __future__ import annotations

from pathlib import Path
import shutil
import subprocess

import numpy as np
import pyvista as pv


# =========================================================
# CONFIG
# =========================================================

OUTPUT_FORMAT = "webm"   # webm | mp4 | gif

FPS = 24
TOTAL_FRAMES = 240

WIDTH = 1920
HEIGHT = 1080

ANIMATION_NAME = "kerr_black_hole"

OUT_DIR = Path("media-site/animations") / ANIMATION_NAME
FRAME_DIR = OUT_DIR / "_frames"

BG_COLOR = "black"

# black hole
BH_RADIUS = 0.72
PHOTON_RING_RADIUS = 1.12

# disk
DISK_INNER = 1.55
DISK_OUTER = 4.8

# frame dragging
FIELD_RADIUS = 8.0
FIELD_LAYERS = 22
FIELD_POINTS = 900

# camera
CAMERA_RADIUS = 16.0
CAMERA_HEIGHT = 5.0

# =========================================================
# HELPERS
# =========================================================

def normalize_output_format(fmt: str) -> str:
    fmt = fmt.lower().strip()

    if fmt not in {"webm", "mp4", "gif"}:
        raise ValueError("OUTPUT_FORMAT must be webm/mp4/gif")

    return fmt


def export_with_ffmpeg(
    frame_dir: Path,
    out_path: Path,
    output_format: str,
):
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg not found. Install with: brew install ffmpeg")

    output_format = normalize_output_format(output_format)

    if output_format == "webm":

        cmd = [
            "ffmpeg",
            "-y",
            "-framerate",
            str(FPS),
            "-i",
            str(frame_dir / "frame_%04d.png"),
            "-c:v",
            "libvpx-vp9",
            "-b:v",
            "0",
            "-crf",
            "28",
            "-pix_fmt",
            "yuva420p",
            "-row-mt",
            "1",
            "-auto-alt-ref",
            "0",
            str(out_path),
        ]

    elif output_format == "mp4":

        cmd = [
            "ffmpeg",
            "-y",
            "-framerate",
            str(FPS),
            "-i",
            str(frame_dir / "frame_%04d.png"),
            "-c:v",
            "libx264",
            "-crf",
            "18",
            "-pix_fmt",
            "yuv420p",
            str(out_path),
        ]

    else:
        palette = frame_dir / "palette.png"

        subprocess.run(
            [
                "ffmpeg",
                "-y",
                "-framerate",
                str(FPS),
                "-i",
                str(frame_dir / "frame_%04d.png"),
                "-vf",
                "palettegen",
                str(palette),
            ],
            check=True,
        )

        cmd = [
            "ffmpeg",
            "-y",
            "-framerate",
            str(FPS),
            "-i",
            str(frame_dir / "frame_%04d.png"),
            "-i",
            str(palette),
            "-lavfi",
            "paletteuse",
            "-loop",
            "0",
            str(out_path),
        ]

    subprocess.run(cmd, check=True)


# =========================================================
# KERR FIELD LINE
# =========================================================

def build_twisted_ring(
    radius: float,
    twist: float,
    z_amp: float,
    phase: float,
):
    t = np.linspace(0, 2 * np.pi, FIELD_POINTS)

    local_twist = twist * (1.0 + 0.2 * np.sin(phase))

    r = radius + 0.22 * np.sin(6 * t + phase)

    x = r * np.cos(t + local_twist / (radius + 0.5))
    y = r * np.sin(t + local_twist / (radius + 0.5))

    z = (
        z_amp * np.sin(2 * t + phase)
        + 0.08 * np.sin(10 * t)
    )

    pts = np.column_stack([x, y, z])

    return pv.Spline(pts, FIELD_POINTS)


# =========================================================
# ACCRETION DISK
# =========================================================

def create_disk():
    disk = pv.Disc(
        inner=DISK_INNER,
        outer=DISK_OUTER,
        r_res=1,
        c_res=240,
    )

    return disk


# =========================================================
# SCENE
# =========================================================

def build_scene(plotter: pv.Plotter, phase: float):

    plotter.clear()

    plotter.set_background(BG_COLOR)

    plotter.enable_anti_aliasing()

    # -----------------------------------------------------
    # BLACK HOLE
    # -----------------------------------------------------

    black_hole = pv.Sphere(
        radius=BH_RADIUS,
        theta_resolution=120,
        phi_resolution=120,
    )

    plotter.add_mesh(
        black_hole,
        color="black",
        smooth_shading=True,
        specular=0.0,
        ambient=0.0,
    )

    # -----------------------------------------------------
    # PHOTON RING
    # -----------------------------------------------------

    photon_ring = pv.ParametricTorus(
        ringradius=PHOTON_RING_RADIUS,
        crosssectionradius=0.035,
    )

    plotter.add_mesh(
        photon_ring,
        color="#ffffff",
        emissive=True,
        smooth_shading=True,
        opacity=0.95,
    )

    # -----------------------------------------------------
    # GLOW
    # -----------------------------------------------------

    for r, opacity in [
        (1.4, 0.08),
        (2.0, 0.04),
    ]:

        glow = pv.Sphere(radius=r)

        plotter.add_mesh(
            glow,
            color="#66e0ff",
            opacity=opacity,
            smooth_shading=True,
        )

    # -----------------------------------------------------
    # ACCRETION DISK
    # -----------------------------------------------------

    disk = create_disk()

    plotter.add_mesh(
        disk,
        color="#ffb366",
        opacity=0.22,
        smooth_shading=True,
        emissive=True,
    )

    # brighter inner disk

    inner_disk = pv.Disc(
        inner=1.4,
        outer=2.5,
        r_res=1,
        c_res=240,
    )

    plotter.add_mesh(
        inner_disk,
        color="#fff2c2",
        opacity=0.55,
        smooth_shading=True,
        emissive=True,
    )

    # -----------------------------------------------------
    # FRAME DRAGGING FIELD
    # -----------------------------------------------------

    for i in range(FIELD_LAYERS):

        frac = i / (FIELD_LAYERS - 1)

        radius = 2.0 + frac * FIELD_RADIUS

        twist = 5.5 * (1.0 - frac)

        z_amp = 0.12 + 0.45 * frac

        spline = build_twisted_ring(
            radius=radius,
            twist=twist,
            z_amp=z_amp,
            phase=phase * 4.0,
        )

        tube = spline.tube(
            radius=0.018 + 0.004 * (1.0 - frac)
        )

        r = 0.2 + 0.5 * (1.0 - frac)
        g = 0.6 + 0.3 * frac
        b = 1.0

        opacity = 0.18 + 0.45 * (1.0 - frac)

        plotter.add_mesh(
            tube,
            color=(r, g, b),
            opacity=opacity,
            smooth_shading=True,
            ambient=0.22,
            diffuse=0.9,
            specular=1.0,
            specular_power=20,
        )

    # -----------------------------------------------------
    # POLAR JETS
    # -----------------------------------------------------

    jet = pv.Cylinder(
        center=(0, 0, 0),
        direction=(0, 0, 1),
        radius=0.10,
        height=18,
        resolution=80,
    )

    plotter.add_mesh(
        jet,
        color="#8ffcff",
        opacity=0.18,
        smooth_shading=True,
        emissive=True,
    )

    # -----------------------------------------------------
    # STAR FIELD
    # -----------------------------------------------------

    rng = np.random.default_rng(42)

    stars = rng.uniform(-40, 40, size=(2500, 3))

    cloud = pv.PolyData(stars)

    plotter.add_mesh(
        cloud,
        color="white",
        point_size=2,
        render_points_as_spheres=True,
        opacity=0.6,
    )


# =========================================================
# MAIN
# =========================================================

def main():

    output_format = normalize_output_format(OUTPUT_FORMAT)

    OUT_DIR.mkdir(parents=True, exist_ok=True)
    FRAME_DIR.mkdir(parents=True, exist_ok=True)

    print("[START] Kerr black hole")

    plotter = pv.Plotter(
        off_screen=True,
        window_size=(WIDTH, HEIGHT),
    )

    for i in range(TOTAL_FRAMES):

        if i % 24 == 0:
            print(f"[RENDER] frame {i}/{TOTAL_FRAMES}")

        phase = i / TOTAL_FRAMES

        build_scene(plotter, phase)

        # -------------------------------------------------
        # CAMERA ORBIT
        # -------------------------------------------------

        angle = 2 * np.pi * phase

        cam_x = CAMERA_RADIUS * np.cos(angle)
        cam_y = CAMERA_RADIUS * np.sin(angle)
        cam_z = CAMERA_HEIGHT + 0.4 * np.sin(angle * 2)

        plotter.camera_position = [
            (cam_x, cam_y, cam_z),
            (0.0, 0.0, 0.0),
            (0.0, 0.0, 1.0),
        ]

        plotter.camera.view_angle = 32
        plotter.render()


        plotter.render()

        frame_path = FRAME_DIR / f"frame_{i:04d}.png"

        plotter.screenshot(str(frame_path))

    plotter.close()

    out_path = OUT_DIR / f"{ANIMATION_NAME}.{output_format}"

    export_with_ffmpeg(
        FRAME_DIR,
        out_path,
        output_format,
    )

    shutil.rmtree(FRAME_DIR, ignore_errors=True)

    print()
    print(f"[CREATED] {out_path.resolve()}")
    print()


main()

[START] Kerr black hole
[RENDER] frame 0/240
[RENDER] frame 24/240
[RENDER] frame 48/240
[RENDER] frame 72/240
[RENDER] frame 96/240
[RENDER] frame 120/240
[RENDER] frame 144/240
[RENDER] frame 168/240
[RENDER] frame 192/240
[RENDER] frame 216/240


ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib


[CREATED] /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/kerr_black_hole/kerr_black_hole.webm



[out#0/webm @ 0x13560eea0] video:7892KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.204667%
frame=  240 fps= 18 q=28.0 Lsize=    7909KiB time=00:00:10.00 bitrate=6478.8kbits/s speed=0.74x    


# TIDAL DISRUPTION EVENT

In [3]:
from __future__ import annotations

from pathlib import Path
import shutil
import subprocess

import numpy as np
import pyvista as pv


# =========================================================
# CONFIG
# =========================================================

OUTPUT_FORMAT = "webm"  # "webm" | "mp4" | "gif"

FPS = 24
TOTAL_FRAMES = 240

WIDTH = 1920
HEIGHT = 1080

ANIMATION_NAME = "tidal_disruption_event"

OUT_DIR = Path("media-site/animations") / ANIMATION_NAME
FRAME_DIR = OUT_DIR / "_frames"

BG_COLOR = "black"

BH_RADIUS = 0.72
PHOTON_RING_RADIUS = 1.10

STAR_RADIUS = 0.42
PARTICLE_COUNT = 2600

CAMERA_RADIUS = 13.5
CAMERA_HEIGHT = 4.4


# =========================================================
# EXPORT
# =========================================================

def normalize_output_format(fmt: str) -> str:
    fmt = fmt.lower().strip()
    if fmt not in {"webm", "mp4", "gif"}:
        raise ValueError("OUTPUT_FORMAT must be webm/mp4/gif")
    return fmt


def export_with_ffmpeg(frame_dir: Path, out_path: Path, output_format: str):
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg not found. Install with: brew install ffmpeg")

    output_format = normalize_output_format(output_format)

    if output_format == "webm":
        cmd = [
            "ffmpeg", "-y",
            "-framerate", str(FPS),
            "-i", str(frame_dir / "frame_%04d.png"),
            "-c:v", "libvpx-vp9",
            "-b:v", "0",
            "-crf", "28",
            "-pix_fmt", "yuva420p",
            "-row-mt", "1",
            "-auto-alt-ref", "0",
            str(out_path),
        ]

    elif output_format == "mp4":
        cmd = [
            "ffmpeg", "-y",
            "-framerate", str(FPS),
            "-i", str(frame_dir / "frame_%04d.png"),
            "-c:v", "libx264",
            "-crf", "18",
            "-pix_fmt", "yuv420p",
            str(out_path),
        ]

    else:
        palette = frame_dir / "palette.png"

        subprocess.run(
            [
                "ffmpeg", "-y",
                "-framerate", str(FPS),
                "-i", str(frame_dir / "frame_%04d.png"),
                "-vf", "palettegen",
                str(palette),
            ],
            check=True,
        )

        cmd = [
            "ffmpeg", "-y",
            "-framerate", str(FPS),
            "-i", str(frame_dir / "frame_%04d.png"),
            "-i", str(palette),
            "-lavfi", "paletteuse",
            "-loop", "0",
            str(out_path),
        ]

    subprocess.run(cmd, check=True)


# =========================================================
# GEOMETRY
# =========================================================

def make_star_particles(seed: int = 7) -> np.ndarray:
    rng = np.random.default_rng(seed)

    u = rng.normal(size=(PARTICLE_COUNT, 3))
    u /= np.linalg.norm(u, axis=1)[:, None]

    r = rng.random(PARTICLE_COUNT) ** (1 / 3) * STAR_RADIUS

    return u * r[:, None]


STAR_PARTICLES = make_star_particles()


def smoothstep(t: float) -> float:
    t = np.clip(t, 0.0, 1.0)
    return t * t * (3.0 - 2.0 * t)


def tidal_stream_positions(phase: float) -> tuple[np.ndarray, np.ndarray]:
    """
    Returns particle positions and scalar heat values.
    This is an artistic / qualitative TDE model, not a hydrodynamic simulation.
    """

    p = STAR_PARTICLES.copy()

    disruption = smoothstep((phase - 0.18) / 0.42)
    wrap = smoothstep((phase - 0.38) / 0.48)

    # Initial star orbit: approaches from right, passes pericenter, then debris wraps.
    star_center = np.array([
        5.6 - 8.2 * smoothstep(min(phase / 0.55, 1.0)),
        -2.4 + 2.8 * np.sin(phase * np.pi * 0.9),
        0.15 * np.sin(phase * np.pi * 2.0),
    ])

    # Particle ordering along tidal axis.
    tidal_axis = p[:, 0] / STAR_RADIUS
    lateral = np.sqrt(p[:, 1] ** 2 + p[:, 2] ** 2) / STAR_RADIUS

    # Stretch into leading/trailing stream.
    stretch_len = 8.0 * disruption
    p[:, 0] += tidal_axis * stretch_len

    # Thin stream as disruption grows.
    p[:, 1] *= 1.0 - 0.72 * disruption
    p[:, 2] *= 1.0 - 0.55 * disruption

    # Convert part of the stream into spiral fallback.
    s = (tidal_axis + 1.0) * 0.5
    theta = 1.0 + s * 5.8 + wrap * 5.2 + phase * 4.0
    radius = 5.5 - 3.9 * s * wrap

    spiral_x = radius * np.cos(theta)
    spiral_y = radius * np.sin(theta)
    spiral_z = 0.18 * np.sin(theta * 2.0 + phase * 8.0) + p[:, 2] * 0.55

    spiral = np.column_stack([spiral_x, spiral_y, spiral_z])

    # Blend original stretched stream into orbital spiral.
    blend = wrap * smoothstep((s - 0.10) / 0.85)
    stretched = p + star_center

    pos = stretched * (1.0 - blend[:, None]) + spiral * blend[:, None]

    # Infall particles closer to hole become hotter.
    dist = np.linalg.norm(pos[:, :2], axis=1)
    heat = np.clip(1.0 - (dist - 1.0) / 5.0, 0.0, 1.0)
    heat = np.maximum(heat, disruption * (0.2 + 0.8 * s))

    # Slight turbulence.
    rng = np.random.default_rng(123)
    jitter = rng.normal(size=pos.shape) * 0.035 * disruption
    pos += jitter

    return pos, heat


def create_accretion_arc(phase: float) -> pv.PolyData:
    t = np.linspace(0.0, 1.0, 1100)

    theta = 1.2 + t * 9.5 + phase * 5.2
    r = 1.45 + 3.6 * (1.0 - t) ** 1.25

    x = r * np.cos(theta)
    y = r * np.sin(theta)
    z = 0.10 * np.sin(theta * 2.0)

    pts = np.column_stack([x, y, z])

    spline = pv.Spline(pts, 900)
    return spline.tube(radius=0.028)


def create_debris_tail(phase: float) -> pv.PolyData:
    t = np.linspace(0.0, 1.0, 900)

    theta = -0.8 - t * 3.0 + phase * 1.2
    r = 3.4 + t * 5.8

    x = r * np.cos(theta)
    y = r * np.sin(theta)
    z = 0.25 * np.sin(t * np.pi * 3.0 + phase * 4.0)

    pts = np.column_stack([x, y, z])

    spline = pv.Spline(pts, 800)
    return spline.tube(radius=0.020)


# =========================================================
# SCENE
# =========================================================

def add_black_hole(plotter: pv.Plotter):
    black_hole = pv.Sphere(
        radius=BH_RADIUS,
        theta_resolution=128,
        phi_resolution=128,
    )

    plotter.add_mesh(
        black_hole,
        color="black",
        smooth_shading=True,
        ambient=0.0,
        diffuse=0.0,
        specular=0.0,
    )

    photon_ring = pv.ParametricTorus(
        ringradius=PHOTON_RING_RADIUS,
        crosssectionradius=0.035,
    )

    plotter.add_mesh(
        photon_ring,
        color="#ffffff",
        opacity=0.92,
        smooth_shading=True,
        emissive=True,
    )

    for r, opacity in [(1.45, 0.08), (2.1, 0.035)]:
        glow = pv.Sphere(radius=r, theta_resolution=80, phi_resolution=80)
        plotter.add_mesh(
            glow,
            color="#66e0ff",
            opacity=opacity,
            smooth_shading=True,
        )


def add_reference_disk(plotter: pv.Plotter, phase: float):
    disk = pv.Disc(
        inner=1.35,
        outer=4.8,
        r_res=1,
        c_res=260,
    )

    opacity = 0.10 + 0.13 * smoothstep((phase - 0.35) / 0.35)

    plotter.add_mesh(
        disk,
        color="#ff9f45",
        opacity=opacity,
        smooth_shading=True,
        emissive=True,
    )


def add_star_or_debris(plotter: pv.Plotter, phase: float):
    pos, heat = tidal_stream_positions(phase)

    cloud = pv.PolyData(pos)
    cloud["heat"] = heat

    plotter.add_mesh(
        cloud,
        scalars="heat",
        cmap="inferno",
        clim=(0.0, 1.0),
        point_size=5.0,
        render_points_as_spheres=True,
        opacity=0.88,
        show_scalar_bar=False,
        emissive=True,
    )

    # Draw a surviving stellar core early in the sequence.
    core_visibility = 1.0 - smoothstep((phase - 0.30) / 0.25)

    if core_visibility > 0.02:
        core_center = np.array([
            5.6 - 8.2 * smoothstep(min(phase / 0.55, 1.0)),
            -2.4 + 2.8 * np.sin(phase * np.pi * 0.9),
            0.15 * np.sin(phase * np.pi * 2.0),
        ])

        core = pv.Sphere(
            radius=STAR_RADIUS * (0.9 - 0.35 * smoothstep((phase - 0.2) / 0.35)),
            center=core_center,
            theta_resolution=64,
            phi_resolution=64,
        )

        plotter.add_mesh(
            core,
            color="#fff2b0",
            opacity=0.55 * core_visibility,
            smooth_shading=True,
            emissive=True,
        )


def add_streams(plotter: pv.Plotter, phase: float):
    stream_visibility = smoothstep((phase - 0.28) / 0.28)

    if stream_visibility > 0.02:
        arc = create_accretion_arc(phase)

        plotter.add_mesh(
            arc,
            color="#ffcc88",
            opacity=0.62 * stream_visibility,
            smooth_shading=True,
            emissive=True,
        )

        tail = create_debris_tail(phase)

        plotter.add_mesh(
            tail,
            color="#ff7040",
            opacity=0.38 * stream_visibility,
            smooth_shading=True,
            emissive=True,
        )


def add_starfield(plotter: pv.Plotter):
    rng = np.random.default_rng(42)
    stars = rng.uniform(-45, 45, size=(1800, 3))
    cloud = pv.PolyData(stars)

    plotter.add_mesh(
        cloud,
        color="white",
        point_size=1.6,
        render_points_as_spheres=True,
        opacity=0.45,
    )


def build_scene(plotter: pv.Plotter, phase: float):
    plotter.clear()
    plotter.set_background(BG_COLOR)
    plotter.enable_anti_aliasing()

    add_starfield(plotter)
    add_reference_disk(plotter, phase)
    add_streams(plotter, phase)
    add_star_or_debris(plotter, phase)
    add_black_hole(plotter)


# =========================================================
# MAIN
# =========================================================

def main():
    output_format = normalize_output_format(OUTPUT_FORMAT)

    OUT_DIR.mkdir(parents=True, exist_ok=True)
    FRAME_DIR.mkdir(parents=True, exist_ok=True)

    print("[START] tidal disruption event")
    print(f"[CONFIG] OUTPUT_FORMAT = {output_format}")
    print(f"[CONFIG] FPS = {FPS}")
    print(f"[CONFIG] TOTAL_FRAMES = {TOTAL_FRAMES}")

    plotter = pv.Plotter(
        off_screen=True,
        window_size=(WIDTH, HEIGHT),
    )

    for i in range(TOTAL_FRAMES):
        if i % 24 == 0:
            print(f"[RENDER] frame {i}/{TOTAL_FRAMES}")

        phase = i / TOTAL_FRAMES

        build_scene(plotter, phase)

        angle = 0.55 + 2 * np.pi * phase * 0.55

        cam_x = CAMERA_RADIUS * np.cos(angle)
        cam_y = CAMERA_RADIUS * np.sin(angle)
        cam_z = CAMERA_HEIGHT + 0.5 * np.sin(2 * np.pi * phase)

        plotter.camera_position = [
            (cam_x, cam_y, cam_z),
            (0.0, 0.0, 0.0),
            (0.0, 0.0, 1.0),
        ]

        plotter.camera.view_angle = 34
        plotter.render()

        frame_path = FRAME_DIR / f"frame_{i:04d}.png"
        plotter.screenshot(str(frame_path))

    plotter.close()

    out_path = OUT_DIR / f"{ANIMATION_NAME}.{output_format}"

    export_with_ffmpeg(
        FRAME_DIR,
        out_path,
        output_format,
    )

    shutil.rmtree(FRAME_DIR, ignore_errors=True)

    print()
    print(f"[CREATED] {out_path.resolve()}")
    print()


main()

[START] tidal disruption event
[CONFIG] OUTPUT_FORMAT = webm
[CONFIG] FPS = 24
[CONFIG] TOTAL_FRAMES = 240
[RENDER] frame 0/240
[RENDER] frame 24/240
[RENDER] frame 48/240
[RENDER] frame 72/240
[RENDER] frame 96/240
[RENDER] frame 120/240
[RENDER] frame 144/240
[RENDER] frame 168/240
[RENDER] frame 192/240
[RENDER] frame 216/240


ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib


[CREATED] /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/tidal_disruption_event/tidal_disruption_event.webm

